# Day 05 — G-Eval: Write Your Own Evaluation Criteria

**Module 1 · Foundations**

So far, we have used a built-in metric.

But what if the quality we want to measure is something specific to our application?

For example:

- Is the response professional?
- Is it empathetic?
- Does it follow our brand voice?
- Is it concise?

G-Eval allows us to define evaluation criteria using natural language.

> **Core idea:** With G-Eval, we can describe what "good" means in natural language and let an LLM judge evaluate it.

## 1. Setup

We will use the same judge model introduced in Day 03.

In [2]:
import os

from dotenv import load_dotenv
from deepeval.models import LocalModel

load_dotenv()

assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found."

judge = LocalModel(
    model="openai/gpt-oss-120b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)

print("Judge:", judge.get_model_name())

Judge: openai/gpt-oss-120b (Local Model)


## 2. Create Our Test Cases

We will use two responses to the same question:

- One professional response.
- One rude and dismissive response.

This will help us see whether our custom metric can distinguish between them.

In [3]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input="What's your refund policy?",
        actual_output=(
            "Happy to help! We offer full refunds within 30 days of purchase. "
            "Just reply with your order number and we'll help you with the process."
        ),
    ),
    LLMTestCase(
        input="What's your refund policy?",
        actual_output=(
            "No refunds. Read the policy page next time."
        ),
    ),
]

for i, test_case in enumerate(test_cases, start=1):
    print(f"Test Case {i}")
    print("Input :", test_case.input)
    print("Output:", test_case.actual_output)
    print()

Test Case 1
Input : What's your refund policy?
Output: Happy to help! We offer full refunds within 30 days of purchase. Just reply with your order number and we'll help you with the process.

Test Case 2
Input : What's your refund policy?
Output: No refunds. Read the policy page next time.



## 3. Create a G-Eval Metric

Let's define our own metric:

**Professional Tone**

Instead of writing evaluation logic in Python, we describe the quality criteria in natural language.

We can provide:

- `name` — name of the metric
- `evaluation_steps` — instructions for the judge
- `evaluation_params` — which test-case fields the judge should evaluate
- `model` — the LLM judge

In [4]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

professional_tone = GEval(
    name="Professional Tone",
    evaluation_steps=[
        "Check whether the response is polite and professional.",
        "Penalize rude, sarcastic, dismissive, or disrespectful language.",
        "Reward clear and respectful communication.",
    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
    ],
    model=judge,
)

print("Metric:", professional_tone.name)

Metric: Professional Tone


## 4. Run the Custom Evaluation

Our evaluation now looks like:

```text
LLMTestCase
     ↓
G-Eval
     ↓
Evaluation Criteria
     ↓
LLM Judge
     ↓
Score + Reason

In [5]:
from deepeval import evaluate

results = evaluate(
    test_cases=test_cases,
    metrics=[professional_tone],
)

for result in results.test_results:
    metric_result = result.metrics_data[0]

    print(f"\n{'=' * 50}")
    print(f"Test Case: {result.name}")
    print(f"Score:     {metric_result.score:.2f}")
    print(f"Success:   {metric_result.success}")
    print(f"Reason:    {metric_result.reason}")

✨ You're running DeepEval's latest Professional Tone [GEval] Metric! (using openai/gpt-oss-120b (Local Model), 
strict=False, async_mode=True)...

c:\Users\T14s\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            What's your refund policy?                                                             │
│  │     Actual Output:    No refunds. Read the policy page next time.                                            │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                    ┃ Score ┃ Threshold ┃ Reason                                           │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Professional Tone [GEval] │ 0.20  │ 0.50      │ The reply is brief but dismissive and not        │
│              │                           │       │           │ politely phrased; saying "Read the policy page   │
│              │                           │       │           │ next time" comes across as rude rather than      │
│              │                           │       │           │ respectful, failing the politeness and           │
│              │                           │       │           │ professionalism criteria.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                              ┃ Average Score       ┃ Pass Rate                               ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Professional Tone [GEval]           │ 0.60                │ 50.00% | passed=1 | failed=1            │ 2        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=446646;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.23s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 50.0% | Passed: 1 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


Test Case: test_case_0
Score:     1.00
Success:   True
Reason:    The response is polite and professional, using friendly language (“Happy to help!”) and clearly explains the refund policy and next steps, meeting the criteria for respectful and clear communication.

Test Case: test_case_1
Score:     0.20
Success:   False
Reason:    The reply is brief but dismissive and not politely phrased; saying "Read the policy page next time" comes across as rude rather than respectful, failing the politeness and professionalism criteria.


## 5. What Did We Build?

We just created a metric without writing a custom scoring algorithm.

We described:

```text
"What does professional mean?"

## 6. G-Eval With a Rubric

Sometimes "good" and "bad" are not enough.

We may want a more detailed grading scale.

For example:

```text
0–2  → Incorrect
3–5  → Partially correct
6–8  → Mostly correct
9–10 → Fully correct

In [6]:
from deepeval.metrics.g_eval import Rubric

correctness = GEval(
    name="Correctness",
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    rubric=[
        Rubric(
            score_range=(0, 2),
            expected_outcome="Factually wrong or contradicts the expected answer.",
        ),
        Rubric(
            score_range=(3, 5),
            expected_outcome="Partially correct but misses an important fact.",
        ),
        Rubric(
            score_range=(6, 8),
            expected_outcome="Mostly correct with minor omissions.",
        ),
        Rubric(
            score_range=(9, 10),
            expected_outcome="Fully correct and complete.",
        ),
    ],
    model=judge,
)

## 7. Why Do Evaluation Parameters Matter?

Notice that the **Professional Tone** metric uses:

```text
INPUT
ACTUAL_OUTPUT

# Day 05 — Key Takeaways

- Built-in metrics are useful, but every application has unique quality requirements.
- **G-Eval** lets us define evaluation criteria using natural language.
- `evaluation_steps` describe how the judge should evaluate the response.
- `evaluation_params` control which test-case fields are used.
- A **rubric** can define different quality levels for different score ranges.
- G-Eval is especially useful for qualities such as tone, style, correctness, empathy, and other application-specific requirements.